In [52]:
MAX_DEPTH = 1
BEST_CLUSTER = 6

In [53]:
# Load cluster

import pandas as pd
import numpy as np

df = pd.read_parquet(
    f"../logs/kmean/clustered_data_gen1_depth{MAX_DEPTH}.parquet"
)

# Outputs rows, columns
df.shape

(8821, 112)

In [54]:
# Quick summary of cluster composition
cluster_summary = df.groupby("cluster").agg(
    rows=("ticker", "count"),
    unique_tickers=("ticker", "nunique"),
    unique_sectors=("sector", "nunique")
)

# Add number of rows per year
year_counts = (
    df.assign(year=df["date"].dt.year)
      .groupby(["cluster", "year"])
      .size()
      .unstack(fill_value=0)
)

# Combine
cluster_summary = cluster_summary.join(year_counts)

cluster_summary.sort_values("rows", ascending=False)

,rows,unique_tickers,unique_sectors,2020
cluster,,,,
6,1947,1947,7,1947
7,1644,1644,5,1644
2,1632,1632,7,1632
4,1068,1068,3,1068
9,879,879,4,879
1,743,743,6,743
5,391,137,1,391
8,322,108,1,322
3,167,167,8,167


In [55]:
# Temp
df.groupby("cluster")[[
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y"
]].median()

,future_ret_1y,future_ret_3y,future_ret_5y
cluster,,,
0,0.231757,-0.699556,-0.740942
1,0.661578,0.392104,0.460432
2,0.617843,0.466138,0.627480
3,0.443333,0.266596,0.418009
4,0.267933,0.278887,0.512415
5,0.902198,2.062638,2.183823
6,0.719146,0.584055,0.948383
7,0.930522,0.681756,1.110636
8,0.118899,0.165902,0.242467


In [56]:
# Look at cluster VS average overall
TARGET_COLS = [
    "future_ret_1d",
    "future_ret_1w",
    "future_ret_1m",
    "future_ret_6m",
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y"
]

ID_COLS = ["ticker", "date", "sector"]

PRIM_EXCLUDE = {
    "price",
    "sector_size",
    "rows",
    "sector_size_market",
    "rows_market"
}

SEC_EXCLUDE = {
    "excess_ret_1y", "excess_ret_5y",
    "trend_vs_sector_1y", "trend_vs_sector_5y",
    "drawdown_rel_1y", "drawdown_rel_5y",
    "excess_vs_market_1y", "trend_vs_market_1y",
    "risk_adjusted_1y", "risk_adjusted_5y",
    "quality_score",
    "sector_Unknown"
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_cols.remove("cluster")

feature_cols = [
    c for c in numeric_cols
    if c not in (
        set(ID_COLS)
        | set(TARGET_COLS)
        | PRIM_EXCLUDE
        | SEC_EXCLUDE
    )
]

overall = df[feature_cols].mean()

cluster = df.groupby("cluster")[feature_cols].mean()

for c in cluster.index:
    print("\nCLUSTER", c)

    diff = (
        cluster.loc[c] - overall
    ).abs().sort_values(ascending=False)

    print(diff.head(15))


CLUSTER 0
pe                           28.897173
ret_5y                        3.145883
ret_3y                        1.933482
ret_1y                        1.573115
beta_1y                       1.389868
ret_6m                        1.352139
cluster_prob_0                0.996826
ret_1m                        0.675078
log_market_cap                0.456352
sector_Healthcare             0.300464
sector_Financial Services     0.278993
sec_ret_5y_dispersion         0.250645
cluster_prob_6                0.220746
ret_1w                        0.217910
cluster_prob_7                0.186449
dtype: float64

CLUSTER 1
pe                           9.388702
cluster_prob_1               0.914361
sec_ret_5y_dispersion        0.327846
sector_Technology            0.307774
sector_Financial Services    0.278993
sector_Healthcare            0.268307
log_market_cap               0.249038
cluster_prob_6               0.219315
ret_3y                       0.207321
ret_5y                       0.20445

In [57]:
# Compare stock performance

cluster_performance = df.groupby("cluster").agg(
    stocks=("ticker","nunique"),
    rows=("ticker","count"),
    median_1y=("future_ret_1y","median"),
    median_3y=("future_ret_3y","median"),
    median_5y=("future_ret_5y","median"),
    positive_5y=("future_ret_5y", lambda x: (x > 0).mean())
)

cluster_performance.sort_values("median_5y", ascending=False)

,stocks,rows,median_1y,median_3y,median_5y,positive_5y
cluster,,,,,,
5,137,391,0.902198,2.062638,2.183823,0.966752
7,1644,1644,0.930522,0.681756,1.110636,0.897202
6,1947,1947,0.719146,0.584055,0.948383,0.872625
9,879,879,0.841937,0.597473,0.728630,0.757679
2,1632,1632,0.617843,0.466138,0.627480,0.726103
4,1068,1068,0.267933,0.278887,0.512415,0.852996
1,743,743,0.661578,0.392104,0.460432,0.686406
3,167,167,0.443333,0.266596,0.418009,0.748503
8,108,322,0.118899,0.165902,0.242467,0.717391


In [58]:
# Top 50 stocks in best cluster
best_cluster = BEST_CLUSTER

best = df[
    df["cluster"] == best_cluster
]

stocks = (
    best.groupby("ticker")
    .agg(
        observations=("ticker","count"),
        avg_future_5y=("future_ret_5y","mean"),
        median_future_5y=("future_ret_5y","median"),
        avg_return_1y=("future_ret_1y","mean"),
        avg_beta=("beta_1y","mean"),
        avg_market_cap=("log_market_cap","mean")
    )
    .sort_values(
        "median_future_5y",
        ascending=False
    )
)

stocks.head(50)

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,
BBW,1,34.611350,34.611350,4.983606,1.271453,16.362653
MOD,1,24.983607,24.983607,3.842623,1.437699,18.897462
DAC,1,23.277318,23.277318,11.953004,1.996217,17.875865
ESEA,1,21.489260,21.489260,3.658767,0.846318,16.022183
BXC,1,17.639902,17.639902,8.535279,1.738697,17.280798
DDS,1,15.049522,15.049522,2.508786,1.668915,19.330900
UAN,1,14.287739,14.287739,3.385871,2.401832,17.616078
HOV,1,13.967002,13.967002,14.170732,2.964760,17.380799
SAABF,1,13.529534,13.529534,0.586402,0.665450,21.020817


In [59]:
# Cluster stability

cluster_frequency = (
    df.groupby("ticker")["cluster"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
)

cluster_frequency.value_counts()

best_stocks = cluster_frequency[
    cluster_frequency == best_cluster
]

best_stocks.head()

Series([], Name: cluster, dtype: int64)

In [60]:
# Winning cluster VS market

comparison = pd.DataFrame({
    "winning_cluster": df[df.cluster==best_cluster]["future_ret_5y"],
    "all_stocks": df["future_ret_5y"]
})

comparison.describe()

,winning_cluster,all_stocks
count,1947.000000,8821.000000
mean,1.454378,1.336341
std,2.168695,2.399808
min,-0.990833,-0.998295
25%,0.316991,0.158593
50%,0.948383,0.777199
75%,1.874903,1.710897
max,34.611350,43.518393
